In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv()

model = ChatGroq(
    model="openai/gpt-oss-20b"
)

class schema(BaseModel):
    sentimet: Literal["positive", "negative"] = Field(
        description="Sentiment of the review"
    )

class DiagonosisSchema(BaseModel):
    category: Literal["Bug", "Support", "Other"] = Field(
        description="The category of issue mentioned in the review"
    )

    emotion: Literal["angry", "disappointed", "calm"] = Field(
        description="The emotional tone expressed by the user"
    )

    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue appears to be"
    )

sutructure_model = model.with_structured_output(
    schema,
    method="json_schema"
)

structured_model2 = model.with_structured_output(
    DiagonosisSchema
)

prompt = "What is the sentiment of the review? The model of mobile is worst"

sutructure_model.invoke(prompt)

class Review(TypedDict):
    review: str
    sentiment: Literal["positive", "negative"]
    diagonosis: dict
    response: str

def sentiment(state: Review):

    prompt = f'Find the sentiment of the review: {state["review"]}'

    sentiment = sutructure_model.invoke(prompt)

    return {
        "sentiment": sentiment.sentimet
    }

def check_sentiment(
    state: Review
) -> Literal["positive_response", "run_diagonosis"]:

    if state["sentiment"] == "positive":
        return "positive_response"

    else:
        return "run_diagonosis"

def positive_response(state: Review):

    prompt = f"""
    Write a warm thank you message in response to this review:

    {state["review"]}

    Also request the user to write feedback.
    """

    value = model.invoke(prompt)

    return {
        "response": value.content
    }

def run_diagonosis(state: Review):

    prompt = f"""
    Diagnose this negative review:

    {state["review"]}

    Also identify the category, emotion and urgency.
    """

    response = structured_model2.invoke(prompt)

    return {
        "diagonosis": response.model_dump()
    }

def negative_response(state: Review):

    diagonosis = state["diagonosis"]

    prompt = f"""
    You are a support assistant.

    The user had a '{diagonosis["category"]}' issue,
    sounded '{diagonosis["emotion"]}',
    and the urgency is '{diagonosis["urgency"]}'.

    Write an empathetic and helpful resolution message.
    """

    response = model.invoke(prompt).content

    return {
        "response": response
    }

graph = StateGraph(Review)

graph.add_node("sentiment", sentiment)
graph.add_node("positive_response", positive_response)
graph.add_node("run_diagonosis", run_diagonosis)
graph.add_node("negative_response", negative_response)

graph.add_edge(START, "sentiment")

graph.add_conditional_edges(
    "sentiment",
    check_sentiment
)

graph.add_edge("positive_response", END)

graph.add_edge(
    "run_diagonosis",
    "negative_response"
)

graph.add_edge("negative_response", END)

work_flow = graph.compile()

initial_state = {
    "review": """
    I have been using this app about a month and know i must say
    that the user interface is very unprofessional and difficult.
    Everything is exactly bad .
    It's very bad experience.
    """
}

result = work_flow.invoke(initial_state)

print(result)

{'review': "\n    I have been using this app about a month and know i must say\n    that the user interface is very unprofessional and difficult.\n    Everything is exactly bad .\n    It's very bad experience.\n    ", 'sentiment': 'negative', 'diagonosis': {'category': 'Support', 'emotion': 'disappointed', 'urgency': 'medium'}, 'response': 'Hi there,\n\nI’m really sorry to hear you’re having trouble—thank you for reaching out. I understand how frustrating it can be when things don’t work the way you expect, and I’m here to help get this sorted out for you as quickly as possible.\n\nCould you share a bit more detail about what’s happening? For example:\n\n- The exact error message (if any)  \n- When the issue first appeared  \n- Any recent changes you made (e.g., software updates, new devices)  \n- What you’ve already tried to fix it\n\nOnce I have a clearer picture, I can walk you through some targeted troubleshooting steps or, if needed, pass the issue on to our engineering team for a